# N-armed Bandits: epsilon-greedy, tuned by Qorotrain

An epsilon-greedy agent plays `n_arms` slot machines for `n_plays` plays. The
result is averaged over `n_repeats` independent bandit problems so that one
lucky draw does not decide the score.

**Qorotrain contract**

* The cell tagged `parameters` holds only plain assignments. Qorotrain (through
  Papermill) injects new values right after it, so nothing may be computed there.
* `tracker.log(play, avg_reward=...)` streams the average reward after every play.
  `examples/bandits.yaml` scores a run by the **last** value (`aggregate: last`).
* Run on its own (outside `qorotrainctl`), the Tracker does nothing and the
  notebook still works.

In [ ]:
import numpy as np

from qorotrain import Tracker

In [ ]:
eps = 0.2         # exploration rate: the parameter Qorotrain tunes
n_plays = 1000    # plays per bandit problem
n_arms = 10       # slot machines per problem
n_repeats = 20    # independent problems averaged into the score
seed = 0          # makes every run reproducible

### Environment

Each repeat is a separate bandit problem: one win probability per arm, drawn
from its own seeded generator. Every run with the same `seed` faces exactly the
same problems, so runs differ only by `eps`.

In [ ]:
rngs = [np.random.default_rng([seed, r]) for r in range(n_repeats)]
probs = np.array([rng.random(n_arms) for rng in rngs])  # win probability of each arm, per repeat

In [ ]:
def get_reward(prob, rng, n=10):
    """Pull an arm: the number of wins in n tries, each won with probability prob."""
    return int((rng.random(n) < prob).sum())

### Update Rule for an Arm

$$
\mu_{\text{new}} = \frac{k\mu_{\text{old}} + x}{k+1}
$$

Where:

* $\mu_{\text{new}}$: updated mean reward
* $\mu_{\text{old}}$: current estimated mean reward
* $k$: number of values observed
* $x$: newly observed reward


In [ ]:
def update_record(record, arm_idx, reward):
    n = record[arm_idx][0]
    mean = record[arm_idx][1]

    new_mean = (n * mean + reward) / (n + 1)

    record[arm_idx][0] += 1
    record[arm_idx][1] = new_mean

    return record


def get_best_arm(record):
    return np.argmax(record[:, 1], axis=0)

### Epsilon Greedy Policy

In [ ]:
def epsilon_greedy(record, eps, rng):
    if rng.random() <= eps:
        # explore
        return int(rng.integers(n_arms))
    else:
        # exploit
        return get_best_arm(record)

### Play and report

`avg_reward` is the mean reward per play so far, averaged over the repeats.
`with Tracker()` sends any buffered points even if the cell fails.

In [ ]:
records = np.zeros((n_repeats, n_arms, 2))  # per repeat: arm -> [k, current mean]
running = np.zeros(n_repeats)               # mean reward per play so far, per repeat

with Tracker() as tracker:
    for play in range(n_plays):
        for r in range(n_repeats):
            arm = epsilon_greedy(records[r], eps, rngs[r])
            reward = get_reward(probs[r, arm], rngs[r])
            update_record(records[r], arm, reward)
            running[r] += (reward - running[r]) / (play + 1)
        avg_reward = float(running.mean())
        tracker.log(play, avg_reward=avg_reward)

In [ ]:
print(f"eps={eps}: average reward {avg_reward:.4f} after {n_plays} plays, over {n_repeats} repeats")